# Phase 6 — evaluate the saved checkpoints (no training)

Runs the Phase 6 experiment arms on the 36 checkpoints trained on Windows (config `bd4e108527`, kinetic `K0`).

* **Needs** `phase6-eval-only-bd4e108527-K0.zip` (or the full `phase6-standalone-artifacts-…zip`) anywhere under *MyDrive*. The checkpoints-only zip is not enough: the arms need the test shards.
* **Never trains.** 29/36 checkpoints are *budget-bound* (best epoch = last epoch of the 40-epoch cap). They are evaluated with `allow_budget_bound=True`, so every report is marked `exploratory` and its identifier ends in `-budget-bound`. The saved convergence metadata is not modified.
* **Survives disconnects.** Each arm is saved to Drive as soon as it finishes; re-running the notebook skips finished arms.

Run the cells top to bottom. After a runtime restart just run them all again.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, subprocess, zipfile, importlib.util

IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
OPTIONS = json.loads(os.environ.get("SPNO_OPTIONS", "{}"))
REPO = Path("/content/pin")
PROJECT_ROOT = Path(os.environ.get("SPNO_PROJECT_ROOT", REPO / "spno"))
SOURCE_ROOT = Path(os.environ.get("SPNO_SOURCE_ROOT", PROJECT_ROOT / "results/phase6-standalone-artifacts"))
OUTPUT_ROOT = Path(os.environ.get("SPNO_OUTPUT_ROOT", "/content/drive/MyDrive/spno/phase6-evaluation"))

# Archives that carry the test shards, richest first. SHA-256 recorded on the Windows original.
KNOWN = {
    "phase6-standalone-artifacts-bd4e108527-K0.zip": "01fd894349dc36dd90386d877693e51bbe3d2d28e98609ccefdf02608a0a0812",
    "phase6-eval-only-bd4e108527-K0.zip": "cc882810f6fec63f36d6923833c05c6dcde5b6d50b2a1df296634be755b1235d",
}

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not REPO.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/omer2822/pin.git", str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only"])
    print(subprocess.check_output(["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True))
    if not (SOURCE_ROOT / "checkpoints").exists():
        found = None
        for name, expected in KNOWN.items():
            hits = list(Path("/content/drive/MyDrive").rglob(name))
            if hits:
                found = (hits[0], expected)
                break
        assert found, "Upload phase6-eval-only-bd4e108527-K0.zip to MyDrive (drive.google.com, not the Colab Files pane)"
        archive, expected = found
        print("archive:", archive)
        assert sha256(archive) == expected, "archive checksum mismatch; re-upload it"
        prefix = "spno/results/phase6-standalone-artifacts/"
        with zipfile.ZipFile(archive) as z:
            assert all(m.startswith(prefix) for m in z.namelist()), "unexpected archive layout"
            z.extractall(REPO)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)])

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("source :", SOURCE_ROOT)
print("output :", OUTPUT_ROOT)

## 1 — Verify every artifact against the Windows hashes
A file that is present but different is an error; `train.pt` files are expected to be absent from the eval-only archive.

In [ ]:
manifest_path = PROJECT_ROOT / "results/phase6-diagnosis-2026-09-22/local-artifacts-manifest.json"
if IN_COLAB:
    manifest = json.loads(manifest_path.read_text())
    ok, absent, failed = [], [], []
    for item in manifest["artifacts"]:
        path = REPO / item["path"]
        if not path.is_file():
            absent.append(item["path"])
        elif path.stat().st_size != item["bytes"] or sha256(path) != item["sha256"]:
            failed.append(item["path"])
        else:
            ok.append(item)
    assert not failed, f"corrupt artifacts: {failed}"
    assert all(p.endswith("train.pt") for p in absent), f"missing evaluation files: {[p for p in absent if not p.endswith('train.pt')]}"
    print(f"verified {len(ok)} files ({sum(i['kind'] == 'checkpoint' for i in ok)} checkpoints); absent (train only): {len(absent)}")
else:
    print("not in Colab: skipping the Windows-manifest check")

## 2 — Inventory

In [ ]:
from spno.workflow import Workflow
from spno.phase_workflow import evaluate_phase
from spno.artifacts import atomic_json

from spno.config import DataConfig
SOURCE_CONFIG = os.environ.get("SPNO_SOURCE_CONFIG")  # tests only; Colab infers DataConfig from the checkpoints
data_config = DataConfig(**json.loads(Path(SOURCE_CONFIG).read_text())["data"]) if SOURCE_CONFIG else None
workflow = Workflow(SOURCE_ROOT, OUTPUT_ROOT, data_config=data_config)
SEEDS = OPTIONS.get("seeds", workflow.seeds)
DEVICE = OPTIONS.get("device", "cpu")  # invariants are measured in float64 on CPU anyway
rows = workflow.inventory()
print(f"{len(rows)} checkpoints, seeds {SEEDS}, data {workflow.data_config.__class__.__name__} quick={workflow.quick}")
print(f"converged: {sum(r['converged'] for r in rows)}   budget-bound: {sum(not r['converged'] for r in rows)}")

## 3 — Sanity check: reproduce the Windows G5 numbers
Recomputes the G5a/G5b dispersion probes for every base checkpoint and compares every number with the frozen-weight results computed on Windows (`frozen-probes/*.json`), matched by checkpoint SHA-256. If this fails, **stop** — something about loading differs and the remaining arms would not be trustworthy.

In [ ]:
import math
from scripts.run_phase6 import _model_from_checkpoint, dispersion_arms
from spno.checkpoints import load_checkpoint_payload

reference_dir = Path(os.environ.get("SPNO_G5_REFERENCE_DIR", PROJECT_ROOT / "results/phase6-diagnosis-2026-09-22/frozen-probes"))
references = {}
for path in reference_dir.glob("base-*.json"):
    record = json.loads(path.read_text())
    references[record["checkpoint_sha256"]] = record

def compare(a, b, where=""):
    """Yield (path, relative difference) for every numeric leaf."""
    if isinstance(a, dict):
        assert set(a) == set(b), f"{where}: keys differ"
        for key in a:
            yield from compare(a[key], b[key], f"{where}/{key}")
    elif isinstance(a, list):
        assert len(a) == len(b), f"{where}: lengths differ"
        for i, (x, y) in enumerate(zip(a, b)):
            yield from compare(x, y, f"{where}[{i}]")
    elif isinstance(a, (int, float)) and not isinstance(a, bool):
        if math.isnan(a) and math.isnan(b):
            return
        yield where, abs(a - b) / max(abs(a), abs(b), 1e-9)
    else:
        assert a == b, f"{where}: {a!r} != {b!r}"

TOLERANCE = 1e-6
checked, diffs = 0, []
for row in rows:
    record = references.get(row["sha256"])
    if record is None or "/" in row["name"]:
        continue
    payload = load_checkpoint_payload(Path(row["path"]))
    model = _model_from_checkpoint(payload, workflow.data_config, expected_name=row["name"])
    model.load_state_dict(payload.state_dict, strict=True)
    tag = record["tag"]
    fresh = json.loads(json.dumps(dispersion_arms({tag: model.eval()}, workflow.data_config.domain, workflow.data_config), default=float))
    diffs += compare(fresh, record["experiments"], f"{tag}/seed{row['seed']}")
    checked += 1
EXPECTED_REFERENCES = int(OPTIONS.get("expected_references", 18 if IN_COLAB else 0))
assert checked == EXPECTED_REFERENCES, f"expected {EXPECTED_REFERENCES} base checkpoints with a Windows reference, matched {checked}"
diffs.sort(key=lambda item: -item[1])
bad = [item for item in diffs if item[1] > TOLERANCE]
print(f"compared {checked} checkpoints, {len(diffs)} numbers; largest relative difference {diffs[0][1] if diffs else 0:.2e}")
for where, diff in bad[:5]:
    print(f"  {diff:.2e}  {where}")
# A lone 'principal' value off by a full period is a branch-cut flip from roundoff, not a loading problem.
assert not bad, f"{len(bad)} numbers differ from Windows by more than {TOLERANCE} — stop and investigate before running the arms"
print("PASS — checkpoints reproduce the Windows measurements")

## 4 — Run the arms, one at a time, saved to Drive
Each finished arm writes `OUTPUT_ROOT/phase6-arms/<arm>.json` pointing at its full report. Finished arms are skipped on re-run. Remove an arm from `ARMS` to skip it; the long ones are the distribution shifts (G1–G4) and the 200-step cascade (G9).

In [ ]:
import time
ARMS = OPTIONS.get("arms", ["G5a", "G5b", "G6b", "G6a", "G7", "G1", "G2", "G3", "G4", "G9"])
index_dir = OUTPUT_ROOT / "phase6-arms"
for arm in ARMS:
    marker = index_dir / f"{arm}.json"
    if marker.exists() and Path(json.loads(marker.read_text())["output"]).is_dir():
        print(f"{arm:4} already done -> {json.loads(marker.read_text())['output']}")
        continue
    started = time.time()
    print(f"{arm:4} running ...", flush=True)
    result = evaluate_phase(workflow, 6, arms=[arm], seeds=SEEDS, device=DEVICE, allow_budget_bound=True)
    atomic_json(marker, {"arm": arm, "output": result["output"], "identifier": result["identifier"],
                         "exploratory": result["exploratory"], "seconds": round(time.time() - started, 1)})
    print(f"{arm:4} done in {time.time() - started:,.0f}s -> {result['output']}", flush=True)

## 5 — Results

In [ ]:
from IPython.display import Image, display
for marker in sorted(index_dir.glob("*.json")):
    entry = json.loads(marker.read_text())
    print(f"{entry['arm']:4} exploratory={entry['exploratory']}  {entry['seconds']:>8}s  {entry['output']}")
    for png in sorted((Path(entry["output"]) / "plots").glob("*.png")):
        display(Image(filename=str(png)))